# Choropleth Map & CHORD Chart - Data Analysis Notebook

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

In [2]:
# Read json files from ../src/json/fips.json
# FIPS 10-4 to Country Name mapping
with open('../src/json/fips.json', 'r') as f:
    fips = json.load(f)

In [3]:
# Loading iso3.json from '../src/json/iso3.json' 
iso3_old = json.load(open('../src/json/iso3_old.json'))
iso3_extension = json.load(open('../src/json/iso3_extension.json'))

iso3 = {**iso3_old, **iso3_extension}

In [4]:
# Function to convert 2-letter FIPS 10-4 country codes to country names
def code_to_country(code):
    try:
        country = fips[code]
        return country
    except:
        raise ValueError(f"Invalid country code: {code}")

In [5]:
# Load  "../data/GDELT/gdelt_keys.parquet"  "../data/GDELT/gdelt_conflict_types.parquet" and "../data/GDELT/gdelt_volume_metrics.parquet" then join on "conflict_id"
# and display the first few rows of the resulting dataframe

gdelt_path = "../data/GDELT/"
df_keys = pd.read_parquet(gdelt_path + "gdelt_keys.parquet")
df_volume = pd.read_parquet(gdelt_path + "gdelt_volume_metrics.parquet")
df_mention_types = pd.read_parquet(gdelt_path + "gdelt_conflict_types.parquet")
df_actor_metrics = pd.read_parquet(gdelt_path + "gdelt_actor_metrics.parquet")

# Join dataframes on conflict_id
df_actors = pd.merge(df_keys, df_volume, on='conflict_id')
df_actors = pd.merge(df_actors, df_mention_types, on='conflict_id')
#df_actors = pd.merge(df_actors, df_actor_metrics, on='conflict_id')
df_actors.head()

,conflict_id,mention_week,conflict_country,actor1_country,actor2_country,media_country,mentions_count,distinct_events,distinct_media_sources,verbal_conflict_mentions,material_conflict_mentions,verbal_conflict_unique_events,material_conflict_unique_events
0,0,2026-01-12,UK,UNK,GBR,UP,16,13,11,3,13,3,10
1,1,2026-01-12,FR,FRA,USA,US,101,28,71,41,60,11,17
2,2,2026-01-12,IS,GBR,UNK,UK,8,8,5,6,2,6,2
3,3,2026-01-12,UP,EUR,RUS,UK,208,5,105,207,1,4,1
4,4,2026-01-12,EZ,SVK,UNK,EZ,6,5,4,5,1,4,1


In [6]:
# find if there are any country codes in df_actors['country_code'] that are not in iso3 keys
missing_codes = set(df_actors['actor1_country']) - set(iso3.keys())

In [7]:
print("Missing country codes:", missing_codes)

Missing country codes: set()


In [8]:
print("Missing country codes actor 2:", set(df_actors['actor2_country']) - set(iso3.keys()))

Missing country codes actor 2: set()


In [9]:
df_actors.head()

,conflict_id,mention_week,conflict_country,actor1_country,actor2_country,media_country,mentions_count,distinct_events,distinct_media_sources,verbal_conflict_mentions,material_conflict_mentions,verbal_conflict_unique_events,material_conflict_unique_events
0,0,2026-01-12,UK,UNK,GBR,UP,16,13,11,3,13,3,10
1,1,2026-01-12,FR,FRA,USA,US,101,28,71,41,60,11,17
2,2,2026-01-12,IS,GBR,UNK,UK,8,8,5,6,2,6,2
3,3,2026-01-12,UP,EUR,RUS,UK,208,5,105,207,1,4,1
4,4,2026-01-12,EZ,SVK,UNK,EZ,6,5,4,5,1,4,1


In [10]:
print(df_actors.columns)

Index(['conflict_id', 'mention_week', 'conflict_country', 'actor1_country',
       'actor2_country', 'media_country', 'mentions_count', 'distinct_events',
       'distinct_media_sources', 'verbal_conflict_mentions',
       'material_conflict_mentions', 'verbal_conflict_unique_events',
       'material_conflict_unique_events'],
      dtype='object')


In [11]:
# dropping column 'distinct_media_sources'
df_actors_filtered = df_actors.copy()
df_actors_filtered = df_actors_filtered.drop(columns=['distinct_media_sources', 'media_country', 'verbal_conflict_mentions', 'verbal_conflict_unique_events', 'conflict_id'])
df_actors_filtered.head()

,mention_week,conflict_country,actor1_country,actor2_country,mentions_count,distinct_events,material_conflict_mentions,material_conflict_unique_events
0,2026-01-12,UK,UNK,GBR,16,13,13,10
1,2026-01-12,FR,FRA,USA,101,28,60,17
2,2026-01-12,IS,GBR,UNK,8,8,2,2
3,2026-01-12,UP,EUR,RUS,208,5,1,1
4,2026-01-12,EZ,SVK,UNK,6,5,1,1


In [12]:
# Finding if every value of fips is in iso3 values and vice versa
fips_values = set(fips.values())
iso3_values = set(iso3.values())
print("FIPS values not in ISO3 values:", fips_values - iso3_values)
print("ISO3 values not in FIPS values:", iso3_values - fips_values)

FIPS values not in ISO3 values: {'Howland Island', 'Glorioso Islands', 'Kingman Reef', 'Spratly Islands', 'Johnston Atoll', 'Coral Sea Islands', 'Juan de Nova Island', 'Clipperton Island', 'Europa Island', 'Paracel Islands', 'Navassa Island', 'Oceans', 'Jarvis Island', 'Akrotiri and Dhekelia', 'Oceania', 'Saint-Barthelemy', 'European Union', 'Jan Mayen', 'Tromelin Island', 'Wake Island', 'Etorofu, Habomai, Shikotan, Kunashiri Islands', 'Ashmore and Cartier Islands', 'Palmyra Atoll'}
ISO3 values not in FIPS values: {'North America', 'South America', 'Persian Gulf', 'Southeast Asia', 'Saint Barthelemy', 'Caribbean', 'South Asia', 'Central Asia', 'Southern Africa', 'Europe', 'Latin America', 'West Africa', 'Africa', 'North Africa', 'The West', 'Asia', 'Eastern Africa', 'Unknown', 'Scandinavia', 'Middle East', 'Åland Islands'}


In [13]:
df_aggregated = df_actors_filtered.groupby(
    ['mention_week', 'conflict_country', 'actor1_country', 'actor2_country']
).agg({
    'mentions_count': 'sum',
    'material_conflict_mentions': 'sum',
    'distinct_events': 'max',
    'material_conflict_unique_events': 'max',
}).reset_index()


In [14]:
df_aggregated.head()

,mention_week,conflict_country,actor1_country,actor2_country,mentions_count,material_conflict_mentions,distinct_events,material_conflict_unique_events
0,2015-02-16,AE,ARE,ARE,39,26,10,7
1,2015-02-16,AE,ARE,UNK,357,197,67,33
2,2015-02-16,AE,ARE,USA,13,0,1,0
3,2015-02-16,AE,DJI,UNK,6,0,2,0
4,2015-02-16,AE,SAU,UNK,23,18,2,1


In [15]:
df_aggregated_by_country = df_aggregated.groupby(
    ['conflict_country', 'actor1_country', 'actor2_country']
).agg({
    'mentions_count': 'sum',
    'material_conflict_mentions': 'sum',
    'distinct_events': 'sum',
    'material_conflict_unique_events': 'sum',
}).reset_index()


In [16]:
df_aggregated_by_country.head()

,conflict_country,actor1_country,actor2_country,mentions_count,material_conflict_mentions,distinct_events,material_conflict_unique_events
0,AA,ABW,ABW,30,15,10,7
1,AA,ABW,CAN,6,4,6,4
2,AA,ABW,CHL,17,17,2,2
3,AA,ABW,ESP,24,24,4,4
4,AA,ABW,GBR,11,0,2,0


In [17]:
# Adding columns 'actor1_country_name' and 'actor2_country_name' and 'conflict_country_name' to df_aggregated_by_country using respectively iso3 mapping for the first two and FIPS for the latter one
df_aggregated_by_country['actor1_country_name'] = df_aggregated_by_country['actor1_country'].map(iso3)
df_aggregated_by_country['actor2_country_name'] = df_aggregated_by_country['actor2_country'].map(iso3)
df_aggregated_by_country['conflict_country_name'] = df_aggregated_by_country['conflict_country'].map(fips)
df_aggregated_by_country.head()

,conflict_country,actor1_country,actor2_country,mentions_count,material_conflict_mentions,distinct_events,material_conflict_unique_events,actor1_country_name,actor2_country_name,conflict_country_name
0,AA,ABW,ABW,30,15,10,7,Aruba,Aruba,Aruba
1,AA,ABW,CAN,6,4,6,4,Aruba,Canada,Aruba
2,AA,ABW,CHL,17,17,2,2,Aruba,Chile,Aruba
3,AA,ABW,ESP,24,24,4,4,Aruba,Spain,Aruba
4,AA,ABW,GBR,11,0,2,0,Aruba,United Kingdom,Aruba


In [18]:
# Picking only rows where conflict_country_name is 'Myanmar'
df_myanmar = df_aggregated_by_country[df_aggregated_by_country['conflict_country_name'] == 'Myanmar']
df_myanmar.head()

,conflict_country,actor1_country,actor2_country,mentions_count,material_conflict_mentions,distinct_events,material_conflict_unique_events,actor1_country_name,actor2_country_name,conflict_country_name
14446,BM,AFG,AFG,18,7,4,2,Afghanistan,Afghanistan,Myanmar
14447,BM,AFG,MMR,20,0,3,0,Afghanistan,Myanmar,Myanmar
14448,BM,AFG,PAK,6,6,2,2,Afghanistan,Pakistan,Myanmar
14449,BM,AFG,UNK,56,23,15,2,Afghanistan,Unknown,Myanmar
14450,BM,AFR,MMR,12,6,7,3,Africa,Myanmar,Myanmar


In [19]:
# Dropping columns 'conflict_country', 'actor1_country', 'actor2_country', 'conflict_country_name'
df_myanmar = df_myanmar.drop(columns=['conflict_country', 'actor1_country', 'actor2_country', 'conflict_country_name'])
df_myanmar.head()

,mentions_count,material_conflict_mentions,distinct_events,material_conflict_unique_events,actor1_country_name,actor2_country_name
14446,18,7,4,2,Afghanistan,Afghanistan
14447,20,0,3,0,Afghanistan,Myanmar
14448,6,6,2,2,Afghanistan,Pakistan
14449,56,23,15,2,Afghanistan,Unknown
14450,12,6,7,3,Africa,Myanmar


In [20]:
df_myanmar_actor1 = df_myanmar.groupby('actor1_country_name').agg({
    'mentions_count': 'sum',
    'material_conflict_mentions': 'sum',
    'distinct_events': 'sum',
    'material_conflict_unique_events': 'sum',
}).reset_index()

df_myanmar_actor1.head()

,actor1_country_name,mentions_count,material_conflict_mentions,distinct_events,material_conflict_unique_events
0,Afghanistan,100,36,24,6
1,Africa,97,77,15,10
2,Albania,22,21,4,3
3,Algeria,19,19,5,5
4,Argentina,17,12,12,7


In [21]:
df_myanmar_actor2 = df_myanmar.groupby('actor2_country_name').agg({
    'mentions_count': 'sum',
    'material_conflict_mentions': 'sum',
    'distinct_events': 'sum',
    'material_conflict_unique_events': 'sum',
}).reset_index()

df_myanmar_actor2.head()

,actor2_country_name,mentions_count,material_conflict_mentions,distinct_events,material_conflict_unique_events
0,Afghanistan,56,36,16,13
1,Africa,56,56,8,8
2,Albania,64,64,3,3
3,Algeria,6,0,3,0
4,Argentina,6,0,3,0


In [22]:
# print the top 10 actor1 countries by mentions_count
print(df_myanmar_actor1.sort_values(by='mentions_count', ascending=False).head(10))

   actor1_country_name  mentions_count  material_conflict_mentions  \
85             Unknown          620763                      392970   
48             Myanmar          338499                      212847   
84       United States           43614                       34229   
9           Bangladesh            4768                        2550   
17               China            3508                        1293   
79            Thailand            3376                        1965   
83      United Kingdom            3204                        2101   
23              Europe            1569                        1144   
30               India            1269                         892   
5            Australia            1161                         761   

    distinct_events  material_conflict_unique_events  
85            91507                            60053  
48            54882                            36295  
84             6015                             4184  
9        

In [23]:
print(df_myanmar_actor2.sort_values(by='mentions_count', ascending=False).head(10))

   actor2_country_name  mentions_count  material_conflict_mentions  \
78             Unknown          743499                      464378   
41             Myanmar          242253                      160272   
77       United States           26417                       19411   
76      United Kingdom            3998                        3027   
8           Bangladesh            3495                        1968   
14               China            2267                        1242   
73            Thailand            2177                        1397   
46         North Korea             982                         772   
63      Southeast Asia             982                         428   
26               India             562                         458   

    distinct_events  material_conflict_unique_events  
78           108231                            69811  
41            42061                            28760  
77             3690                             2635  
76       

In [24]:
print(len(df_myanmar_actor1))
print(len(df_myanmar_actor2))

88
82


In [25]:
# Verify that df_myanmar_actor1.sort_values(by='mentions_count', ascending=False).head(10) is equal to df_myanmar_actor1.sort_values(by='material_conflict_mentions', ascending=False).head(10)
print(df_myanmar_actor1.sort_values(by='mentions_count', ascending=False).head(10))
print('------------------------------------------------------------------------')
print(df_myanmar_actor1.sort_values(by='material_conflict_mentions', ascending=False).head(10))
print('------------------------------------------------------------------------')
print(df_myanmar_actor1.sort_values(by='distinct_events', ascending=False).head(10))
print('------------------------------------------------------------------------')
print(df_myanmar_actor1.sort_values(by='material_conflict_unique_events', ascending=False).head(10))

   actor1_country_name  mentions_count  material_conflict_mentions  \
85             Unknown          620763                      392970   
48             Myanmar          338499                      212847   
84       United States           43614                       34229   
9           Bangladesh            4768                        2550   
17               China            3508                        1293   
79            Thailand            3376                        1965   
83      United Kingdom            3204                        2101   
23              Europe            1569                        1144   
30               India            1269                         892   
5            Australia            1161                         761   

    distinct_events  material_conflict_unique_events  
85            91507                            60053  
48            54882                            36295  
84             6015                             4184  
9        

In [27]:
df_myanmar_actor1.head()

,actor1_country_name,mentions_count,material_conflict_mentions,distinct_events,material_conflict_unique_events
0,Afghanistan,100,36,24,6
1,Africa,97,77,15,10
2,Albania,22,21,4,3
3,Algeria,19,19,5,5
4,Argentina,17,12,12,7


In [28]:
df_myanmar_actor2.head()

,actor2_country_name,mentions_count,material_conflict_mentions,distinct_events,material_conflict_unique_events
0,Afghanistan,56,36,16,13
1,Africa,56,56,8,8
2,Albania,64,64,3,3
3,Algeria,6,0,3,0
4,Argentina,6,0,3,0


In [29]:
# keeping only actor1_country_name and material_conflict_unique_events
df_myanmar_actor1 = df_myanmar_actor1[['actor1_country_name', 'material_conflict_unique_events']]
df_myanmar_actor2 = df_myanmar_actor2[['actor2_country_name', 'material_conflict_unique_events']]

In [30]:
# Renaming columns for clarity
df_myanmar_actor1 = df_myanmar_actor1.rename(columns={'actor1_country_name': 'country_name', 'material_conflict_unique_events': 'actor1_material_conflict_unique_events'})
df_myanmar_actor2 = df_myanmar_actor2.rename(columns={'actor2_country_name': 'country_name', 'material_conflict_unique_events': 'actor2_material_conflict_unique_events'})

In [31]:
# Now merging the two tables on country name
df_myanmar_merged = pd.merge(df_myanmar_actor1, df_myanmar_actor2, on='country_name', how='outer')
print(len(df_myanmar_merged))

99


In [32]:
df_myanmar_merged.head()

,country_name,actor1_material_conflict_unique_events,actor2_material_conflict_unique_events
0,Afghanistan,6.0,13.0
1,Africa,10.0,8.0
2,Albania,3.0,3.0
3,Algeria,5.0,0.0
4,Argentina,7.0,0.0


In [33]:
# Adding a new column 'total_material_conflict_unique_events' which is the sum of actor1_material_conflict_unique_events and actor2_material_conflict_unique_events
df_myanmar_merged['total_material_conflict_unique_events'] = df_myanmar_merged['actor1_material_conflict_unique_events'].fillna(0) + df_myanmar_merged['actor2_material_conflict_unique_events'].fillna(0)

# changing column names to countAsActor1, countAsActor2 and totalCount
df_myanmar_merged = df_myanmar_merged.rename(columns={
    'actor1_material_conflict_unique_events': 'countAsActor1',
    'actor2_material_conflict_unique_events': 'countAsActor2',
    'total_material_conflict_unique_events': 'totalCount'
})

df_myanmar_merged.head()

,country_name,countAsActor1,countAsActor2,totalCount
0,Afghanistan,6.0,13.0,19.0
1,Africa,10.0,8.0,18.0
2,Albania,3.0,3.0,6.0
3,Algeria,5.0,0.0,5.0
4,Argentina,7.0,0.0,7.0


In [34]:
# Sort by totalCount descending
df_myanmar_merged = df_myanmar_merged.sort_values(by='totalCount', ascending=False)

In [35]:
df_myanmar_merged.head()

,country_name,countAsActor1,countAsActor2,totalCount
95,Unknown,60053.0,69811.0,129864.0
54,Myanmar,36295.0,28760.0,65055.0
94,United States,4184.0,2635.0,6819.0
93,United Kingdom,281.0,452.0,733.0
10,Bangladesh,388.0,324.0,712.0


In [36]:
# dropping row with country_name 'Unknown'
df_myanmar_merged = df_myanmar_merged[df_myanmar_merged['country_name'] != 'Unknown']
df_myanmar_merged.head()

,country_name,countAsActor1,countAsActor2,totalCount
54,Myanmar,36295.0,28760.0,65055.0
94,United States,4184.0,2635.0,6819.0
93,United Kingdom,281.0,452.0,733.0
10,Bangladesh,388.0,324.0,712.0
88,Thailand,453.0,235.0,688.0


In [38]:
# Taking the top 20 for totalCount
df_myanmar_top20 = df_myanmar_merged.head(20)
df_myanmar_top20.head()

,country_name,countAsActor1,countAsActor2,totalCount
54,Myanmar,36295.0,28760.0,65055.0
94,United States,4184.0,2635.0,6819.0
93,United Kingdom,281.0,452.0,733.0
10,Bangladesh,388.0,324.0,712.0
88,Thailand,453.0,235.0,688.0


In [41]:
# Csv export to "../data/processed/word_cloud.csv"
df_myanmar_top20.to_csv("../data/processed/myanmar_word_cloud.csv", index=False)

In [47]:
def create_word_cloud_data(df_aggregated_by_country, country, count_col):
    df_country = df_aggregated_by_country[df_aggregated_by_country['conflict_country_name'] == country]
    df_country = df_country.drop(columns=['conflict_country', 'actor1_country', 'actor2_country', 'conflict_country_name'])
    
    df_actor1 = df_country.groupby('actor1_country_name').agg({
        'mentions_count': 'sum',
        'material_conflict_mentions': 'sum',
        'distinct_events': 'sum',
        'material_conflict_unique_events': 'sum',
    }).reset_index()

    df_actor2 = df_country.groupby('actor2_country_name').agg({
        'mentions_count': 'sum',
        'material_conflict_mentions': 'sum',
        'distinct_events': 'sum',
        'material_conflict_unique_events': 'sum',
    }).reset_index()

    df_actor1 = df_actor1[['actor1_country_name', count_col]]
    df_actor2 = df_actor2[['actor2_country_name', count_col]]

    # Renaming columns for clarity
    df_actor1 = df_actor1.rename(columns={'actor1_country_name': 'country_name', count_col: 'actor1_material_conflict_unique_events'})
    df_actor2 = df_actor2.rename(columns={'actor2_country_name': 'country_name', count_col: 'actor2_material_conflict_unique_events'})    

    # Now merging the two tables on country name
    df_merged = pd.merge(df_actor1, df_actor2, on='country_name', how='outer')

    # Adding a new column 'total_material_conflict_unique_events' which is the sum of actor1_material_conflict_unique_events and actor2_material_conflict_unique_events
    df_merged['total_material_conflict_unique_events'] = df_merged['actor1_material_conflict_unique_events'].fillna(0) + df_merged['actor2_material_conflict_unique_events'].fillna(0)

    # changing column names to countAsActor1, countAsActor2 and totalCount
    df_merged = df_merged.rename(columns={
        'actor1_material_conflict_unique_events': 'countAsActor1',
        'actor2_material_conflict_unique_events': 'countAsActor2',
        'total_material_conflict_unique_events': 'totalCount'
    })

    df_merged = df_merged.sort_values(by='totalCount', ascending=False)

    # dropping row with country_name 'Unknown'
    df_merged = df_merged[df_merged['country_name'] != 'Unknown']

    # Taking the top 20 for totalCount
    df_top20 = df_merged.head(20)

    # Csv export to "../data/processed/word_cloud.csv"
    df_top20.to_csv(f"../data/processed/{country.lower().replace(' ', '_')}_word_cloud.csv", index=False)

In [48]:
count_col = 'material_conflict_unique_events'
create_word_cloud_data(df_aggregated_by_country, 'Myanmar', count_col)
create_word_cloud_data(df_aggregated_by_country, 'Burkina Faso', count_col)
create_word_cloud_data(df_aggregated_by_country, 'Palestine', count_col)
